In [1]:
import torch

from torch_openreml import MixedModelREML
from torch_openreml.covariance import DummyMatrix, ScalarMatrix

n, p = 10, 2

group = ["a", "b"] * 5
effect = torch.tensor([1.0, -1.0]).repeat(n // 2)

X = torch.randn(n, p)
y = X @ torch.tensor([0.5, -0.5]) + effect + 0.5 * torch.randn(n)

reml = MixedModelREML(
    genotype=(DummyMatrix(group), ScalarMatrix(group)),
)

names, free_params = reml.optimize(y, X, verbose=1)
names, free_params

[  1] ∥∇∥:       4.7739, ∥Δ∥: 1.3122, η: 0.50, ∥Δᶜ∥: 0.6561, accepted: True , log 𝓛:  -2.1719
[  2] ∥∇∥:       0.4179, ∥Δ∥: 0.0838, η: 1.00, ∥Δᶜ∥: 0.0838, accepted: True , log 𝓛:  -2.1604
[  3] ∥∇∥:       0.0258, ∥Δ∥: 0.0041, η: 1.00, ∥Δᶜ∥: 0.0041, accepted: True , log 𝓛:  -2.1603
[  4] ∥∇∥:       0.0021, ∥Δ∥: 0.0009, η: 1.00, ∥Δᶜ∥: 0.0009, accepted: True , log 𝓛:  -2.1603
[  5] ∥∇∥:       0.0004, ∥Δ∥: 0.0002, η: 0.01, ∥Δᶜ∥: 0.0000, accepted: True , log 𝓛:  -2.1603

[∇: score, Δ: 𝐉⁻¹∇, η: step length, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 5, total time: 0.01s



(['G/sigma^2', 'R/sigma^2'], tensor([-0.3648, -0.6180]))

In [2]:
history = reml.history
history["loglik"]

[tensor(-2.1719),
 tensor(-2.1604),
 tensor(-2.1603),
 tensor(-2.1603),
 tensor(-2.1603)]

In [3]:
torch.stack(history["loglik"])

tensor([-2.1719, -2.1604, -2.1603, -2.1603, -2.1603])

In [4]:
torch.stack(history["free_params"])

tensor([[-0.2814, -0.5927],
        [-0.3605, -0.6205],
        [-0.3639, -0.6182],
        [-0.3648, -0.6180],
        [-0.3648, -0.6180]])

In [5]:
names, theta_hat = reml.get_free_params("last")
names, theta_hat

(['G/sigma^2', 'R/sigma^2'], tensor([-0.3648, -0.6180]))

In [6]:
params = reml.build_params("last", out_format="dict")
params

{'G/sigma^2': tensor([0.4821]), 'R/sigma^2': tensor([0.2906])}

In [7]:
tree = reml.tree("last")
tree

{'/': tensor([[0.7727, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.4821,
          0.0000],
         [0.0000, 0.7727, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000,
          0.4821],
         [0.4821, 0.0000, 0.7727, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.4821,
          0.0000],
         [0.0000, 0.4821, 0.0000, 0.7727, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000,
          0.4821],
         [0.4821, 0.0000, 0.4821, 0.0000, 0.7727, 0.0000, 0.4821, 0.0000, 0.4821,
          0.0000],
         [0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.7727, 0.0000, 0.4821, 0.0000,
          0.4821],
         [0.4821, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.7727, 0.0000, 0.4821,
          0.0000],
         [0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.7727, 0.0000,
          0.4821],
         [0.4821, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.7727,
          0.0000],
         [0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000, 0.4821, 0.0000,
    

In [8]:
beta_hat = reml.blue(y, X, theta_hat)
beta_hat

tensor([ 0.1380, -0.2092])

In [9]:
cov_beta = reml.cov_blue(X, theta_hat)
cov_beta

tensor([[0.0865, 0.0534],
        [0.0534, 0.2026]])

In [10]:
se_beta = torch.sqrt(reml.var_blue(X, theta_hat))
se_beta

tensor([0.2942, 0.4502])

In [11]:
b_hat = reml.blup()
b_hat

tensor([ 0.7254, -0.3754])

In [12]:
b_cov = reml.cov_blup()
b_cov

tensor([[0.4012, 0.0681],
        [0.0681, 0.2661]])

In [13]:
b_var = reml.var_blup()
b_var

tensor([0.4012, 0.2661])

In [14]:
b_by_term = reml.blup_per_term()
b_by_term

{'genotype': tensor([ 0.7254, -0.3754])}

In [15]:
b_cov_by_term = reml.cov_blup_per_term()
b_cov_by_term

{'genotype': tensor([[0.4012, 0.0681],
         [0.0681, 0.2661]])}

In [16]:
b_var_by_term = reml.var_blup_per_term()
b_var_by_term

{'genotype': tensor([0.4012, 0.2661])}

In [17]:
y_marginal = reml.marginal_predict()
y_marginal

tensor([ 0.0842,  0.3563, -0.1774,  0.1645, -0.1519,  0.4565, -0.3185,  0.2278,
         0.0111,  0.3390])

In [18]:
y_conditional = reml.predict()
y_conditional

tensor([ 0.8096, -0.0190,  0.5480, -0.2108,  0.5735,  0.0811,  0.4069, -0.1476,
         0.7365, -0.0364])

In [19]:
y_new = reml.predict_new(
    X[:2],
    {"genotype": torch.tensor([[1.0, 0.0], [0.0, 1.0]])},
)
y_new

tensor([ 0.8096, -0.0190])

In [20]:
z_new = DummyMatrix(group).transform(["a", "b"])

y_new = reml.predict_new(
    X[:2],
    z_new,
)
y_new

tensor([ 0.8096, -0.0190])

In [21]:
e_marginal = reml.marginal_residual(y, X, theta_hat)
e_marginal

tensor([ 1.2514, -0.9525,  0.7147, -1.1608,  0.5374, -0.0401,  1.1091, -0.0230,
         0.4515,  0.0734])

In [22]:
e_conditional = reml.residual(y, X, theta_hat)
e_conditional

tensor([ 0.5260, -0.5772, -0.0107, -0.7855, -0.1880,  0.3353,  0.3837,  0.3524,
        -0.2739,  0.4488])

In [23]:
c11, c12, c21, c22 = reml.c_inverse_blocks(X, theta_hat)
c11, c12, c21, c22

(tensor([[0.0865, 0.0534],
         [0.0534, 0.2026]]),
 tensor([[-0.0123,  0.0083],
         [-0.0757,  0.1717]]),
 tensor([[-0.0123, -0.0757],
         [ 0.0083,  0.1717]]),
 tensor([[ 0.0810, -0.0681],
         [-0.0681,  0.2161]]))

In [24]:
loglik_value, loglik_method = reml.loglik(y, X, theta_hat)
loglik_value, loglik_method

(tensor(-2.1603), 'cform')

In [25]:
reml.is_converged()

True